# GOV-01 V2-E2: Validation Error Analysis

This notebook reviews mistakes made by the completed frozen MobileNetV2 V2-E2 model. It does **not** train, change labels, move images, or load `protected_test`.

It creates a downloadable package containing: a full error CSV, a confusion summary, per-class F1 evidence, and a small image gallery for every class below the 99% F1 target. Error analysis helps choose the next experiment; it does not improve model performance by itself.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import zipfile

ARCHIVE_PATH = Path('/content/drive/MyDrive/GOV-01/v2/multiclass_final.zip')
CHECKPOINT_PATH = Path('/content/drive/MyDrive/GOV-01/v2_e2_checkpoint/v2_e2_frozen_mobilenetv2_best.keras')
DATA_DIR = Path('/content/data/processed/v2/multiclass_final')
OUTPUT_DIR = Path('/content/v2_e2_error_analysis')

if not ARCHIVE_PATH.is_file():
    raise FileNotFoundError(f'Dataset ZIP not found in Google Drive: {ARCHIVE_PATH}')
if not CHECKPOINT_PATH.is_file():
    raise FileNotFoundError(
        f'V2-E2 checkpoint not found in Google Drive: {CHECKPOINT_PATH}\n'
        'Run the corrected V2-E2 notebook first, or restore its checkpoint folder.'
    )
if not DATA_DIR.is_dir():
    with zipfile.ZipFile(ARCHIVE_PATH) as archive:
        archive.extractall(DATA_DIR.parent)
    print('Extracted V2 data from Google Drive.')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Validation data:', DATA_DIR / 'validation')
print('Saved E2 model:', CHECKPOINT_PATH)
print('Temporary analysis output:', OUTPUT_DIR)
print('The protected-test folder is deliberately not loaded.')

In [ ]:
import csv
import html
import json
import shutil

import numpy as np
import tensorflow as tf
from PIL import Image, ImageDraw, ImageOps
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

SEED = 42
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
CLASS_NAMES = ['crack', 'manhole_cover', 'normal_asphalt', 'pothole', 'repaired_road', 'speed_bump', 'unpaved_road']
GALLERY_F1_TARGET = 0.99
MAX_GALLERY_IMAGES_PER_CLASS = 12

tf.keras.utils.set_random_seed(SEED)
print('TensorFlow:', tf.__version__)
print('Gallery target: F1 below', GALLERY_F1_TARGET)

In [ ]:
# Only validation is loaded. The protected test set is never read.
validation_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR / 'validation',
    class_names=CLASS_NAMES,
    label_mode='int',
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False,
)
validation_ds = validation_ds.prefetch(tf.data.AUTOTUNE)

# Build the same deterministic file order used by image_dataset_from_directory.
validation_paths = []
expected_labels = []
for label_index, class_name in enumerate(CLASS_NAMES):
    class_paths = sorted(path for path in (DATA_DIR / 'validation' / class_name).iterdir() if path.is_file())
    validation_paths.extend(class_paths)
    expected_labels.extend([label_index] * len(class_paths))

if len(validation_paths) != 1658:
    raise AssertionError(f'Expected 1658 validation images, found {len(validation_paths)}')
print('Validation images:', len(validation_paths))

In [ ]:
model = tf.keras.models.load_model(CHECKPOINT_PATH)
y_true = np.concatenate([labels.numpy() for _, labels in validation_ds])
if not np.array_equal(y_true, np.asarray(expected_labels)):
    raise AssertionError('Validation file order does not match labels. Stop: no review files were written.')

probabilities = model.predict(validation_ds, verbose=1)
y_pred = probabilities.argmax(axis=1)

print('Validation accuracy:', f'{accuracy_score(y_true, y_pred):.4f}')
print('Validation macro F1:', f'{f1_score(y_true, y_pred, average="macro", zero_division=0):.4f}')

In [ ]:
# Write a row for every incorrect validation prediction.
mistakes = []
for index, (path, true_index, predicted_index) in enumerate(zip(validation_paths, y_true, y_pred)):
    if true_index == predicted_index:
        continue
    mistakes.append({
        'relative_path': path.relative_to(DATA_DIR).as_posix(),
        'true_class': CLASS_NAMES[int(true_index)],
        'predicted_class': CLASS_NAMES[int(predicted_index)],
        'predicted_confidence': round(float(probabilities[index, predicted_index]), 6),
        'true_class_confidence': round(float(probabilities[index, true_index]), 6),
    })

mistakes_path = OUTPUT_DIR / 'v2_e2_validation_misclassifications.csv'
with mistakes_path.open('w', newline='', encoding='utf-8') as handle:
    writer = csv.DictWriter(handle, fieldnames=['relative_path', 'true_class', 'predicted_class', 'predicted_confidence', 'true_class_confidence'])
    writer.writeheader()
    writer.writerows(mistakes)

matrix = confusion_matrix(y_true, y_pred, labels=range(len(CLASS_NAMES)))
confusions = []
for true_index, true_class in enumerate(CLASS_NAMES):
    for predicted_index, predicted_class in enumerate(CLASS_NAMES):
        count = int(matrix[true_index, predicted_index])
        if true_index != predicted_index and count:
            confusions.append({
                'true_class': true_class,
                'predicted_class': predicted_class,
                'count': count,
            })
confusions.sort(key=lambda row: row['count'], reverse=True)

confusions_path = OUTPUT_DIR / 'v2_e2_top_validation_confusions.csv'
with confusions_path.open('w', newline='', encoding='utf-8') as handle:
    writer = csv.DictWriter(handle, fieldnames=['true_class', 'predicted_class', 'count'])
    writer.writeheader()
    writer.writerows(confusions)

report = classification_report(y_true, y_pred, target_names=CLASS_NAMES, output_dict=True, zero_division=0)
class_rows = []
for class_name in CLASS_NAMES:
    metrics = report[class_name]
    class_rows.append({
        'class_name': class_name,
        'precision': round(metrics['precision'], 6),
        'recall': round(metrics['recall'], 6),
        'f1_score': round(metrics['f1-score'], 6),
        'support': int(metrics['support']),
        'meets_99_percent_f1_target': metrics['f1-score'] >= GALLERY_F1_TARGET,
    })

class_metrics_path = OUTPUT_DIR / 'v2_e2_validation_class_metrics.csv'
with class_metrics_path.open('w', newline='', encoding='utf-8') as handle:
    writer = csv.DictWriter(handle, fieldnames=list(class_rows[0]))
    writer.writeheader()
    writer.writerows(class_rows)

review_classes = [row['class_name'] for row in class_rows if row['f1_score'] < GALLERY_F1_TARGET]

summary = {
    'run_name': 'v2_e2_frozen_mobilenetv2_error_analysis',
    'split_used': 'validation only',
    'protected_test_loaded': False,
    'validation_images': len(validation_paths),
    'incorrect_predictions': len(mistakes),
    'validation_accuracy': float(accuracy_score(y_true, y_pred)),
    'validation_macro_f1': float(f1_score(y_true, y_pred, average='macro', zero_division=0)),
    'top_confusions': confusions[:15],
    'class_metrics': class_rows,
    'gallery_f1_target': GALLERY_F1_TARGET,
    'gallery_classes_below_target': review_classes,
}
(OUTPUT_DIR / 'v2_e2_error_analysis_summary.json').write_text(json.dumps(summary, indent=2) + '\n', encoding='utf-8')

print('Incorrect validation predictions:', len(mistakes))
print('Top confusions:')
for row in confusions[:10]:
    print(f"  {row['true_class']} → {row['predicted_class']}: {row['count']}")
print('Classes included in the gallery:', review_classes)

In [ ]:
# Make a small visual gallery: the highest-confidence mistakes for every class below the 99% F1 target.
gallery_dir = OUTPUT_DIR / 'gallery'
gallery_dir.mkdir(exist_ok=True)
gallery_rows = []
for true_class in review_classes:
    selected = sorted(
        (row for row in mistakes if row['true_class'] == true_class),
        key=lambda row: row['predicted_confidence'],
        reverse=True,
    )[:MAX_GALLERY_IMAGES_PER_CLASS]
    for number, row in enumerate(selected, start=1):
        original_path = DATA_DIR / row['relative_path']
        with Image.open(original_path).convert('RGB') as image:
            thumbnail = ImageOps.contain(image, (360, 240))
            card = Image.new('RGB', (380, 300), 'white')
            card.paste(thumbnail, ((380 - thumbnail.width) // 2, 4))
            draw = ImageDraw.Draw(card)
            draw.text((8, 250), f"Actual: {row['true_class']}", fill='black')
            draw.text((8, 268), f"Predicted: {row['predicted_class']} ({row['predicted_confidence']:.1%})", fill='black')
        image_name = f"{true_class}_{number:02d}.jpg"
        card.save(gallery_dir / image_name, quality=90)
        gallery_rows.append({**row, 'gallery_file': f'gallery/{image_name}'})

sections = []
for true_class in review_classes:
    cards = [row for row in gallery_rows if row['true_class'] == true_class]
    card_html = ''.join(
        f"<figure><img src='{html.escape(row['gallery_file'])}'><figcaption>{html.escape(row['relative_path'])}</figcaption></figure>"
        for row in cards
    ) or '<p>No incorrect validation images for this class.</p>'
    sections.append(f'<h2>{html.escape(true_class)}</h2><div class="grid">{card_html}</div>')

gallery_html = '''<!doctype html><html><head><meta charset="utf-8"><title>V2-E2 Error Analysis Gallery</title>
<style>body{font-family:Arial,sans-serif;margin:24px}.grid{display:flex;flex-wrap:wrap;gap:14px}figure{width:380px;margin:0;border:1px solid #ccc;padding:6px}img{width:380px;height:300px;object-fit:contain}figcaption{font-size:12px;overflow-wrap:anywhere}</style>
</head><body><h1>V2-E2 Validation Error Analysis</h1><p>Highest-confidence incorrect predictions for every class below the 99% F1 target. This is validation data only.</p>''' + ''.join(sections) + '</body></html>'
(OUTPUT_DIR / 'v2_e2_error_gallery.html').write_text(gallery_html, encoding='utf-8')
print('Gallery images created:', len(gallery_rows))

## How to review the package

- Open `v2_e2_validation_class_metrics.csv` in Excel to see which classes are below the 99% F1 target.
- Open `v2_e2_top_validation_confusions.csv` to see the most frequent actual → predicted mistakes.
- Open `v2_e2_error_gallery.html` in a browser to view the selected road images. The gallery images are included separately too.
- Do not relabel, delete, move, or alter any dataset image from this review alone. Report the repeated visual patterns first.

In [ ]:
# Download the review package. It is created in temporary Colab storage, not in the Git repository.
from google.colab import files

archive_path = shutil.make_archive('/content/v2_e2_error_analysis', 'zip', OUTPUT_DIR)
files.download(archive_path)